# LLM Confidence & Calibration Experiment
**Research Question**: What structural and semantic properties of questions predict when LLMs will be confidently wrong?

**Runtime**: Change to **T4 GPU** (Runtime > Change runtime type > T4 GPU)

**Total time**: ~30-45 minutes for 3 models x 1000 questions

## 1. Setup

In [ ]:
!pip install -q datasets transformers accelerate torch pandas numpy scikit-learn statsmodels seaborn matplotlib sentence-transformers
!nvidia-smi

In [ ]:
import re
import time
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Configuration

In [ ]:
# ── Config ──
CONFIDENT_THRESHOLD = 80
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 50

MODELS = {
    'qwen-2.5-3b': 'Qwen/Qwen2.5-3B-Instruct',
    'qwen-2.5-1.5b': 'Qwen/Qwen2.5-1.5B-Instruct',
}

PROMPT_TEMPLATE = """Answer the following multiple-choice question. After providing your answer, rate your confidence that your answer is correct on a scale from 0 to 100, where 0 means you are certain you are wrong and 100 means you are certain you are correct.

Question: {question}
{options}

Respond in exactly this format:
Answer: [A/B/C/D]
Confidence: [0-100]"""

LETTER_MAP = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}

# Output directory
OUT = Path('results')
OUT.mkdir(exist_ok=True)
FIG = Path('figures')
FIG.mkdir(exist_ok=True)

## 3. Load Datasets

In [ ]:
# ── MMLU ──
print('Loading MMLU...')
ds_mmlu = load_dataset('cais/mmlu', 'all', split='test')
rows = []
for i, item in enumerate(ds_mmlu):
    choices = item['choices']
    ci = item['answer']
    while len(choices) < 4: choices.append('')
    rows.append({
        'question_id': f'mmlu_{i:05d}', 'dataset': 'mmlu',
        'subject': item.get('subject', 'unknown'),
        'question': item['question'],
        'option_a': choices[0], 'option_b': choices[1],
        'option_c': choices[2], 'option_d': choices[3],
        'correct_answer': choices[ci], 'correct_letter': LETTER_MAP[ci],
    })
mmlu = pd.DataFrame(rows)
print(f'  MMLU: {len(mmlu)} questions, {mmlu["subject"].nunique()} subjects')

# ── ARC-Challenge ──
print('Loading ARC-Challenge...')
ds_arc = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
rows = []
lmap = {'1':'A','2':'B','3':'C','4':'D'}
for i, item in enumerate(ds_arc):
    labels = item['choices']['label']
    texts = item['choices']['text']
    l2t = dict(zip(labels, texts))
    opts = []
    for L in ['A','B','C','D']:
        if L in l2t: opts.append(l2t[L])
        elif lmap.get(L,'') in l2t: opts.append(l2t[lmap[L]])
        else: break
    if len(opts) != 4: continue
    ak = item['answerKey']
    if ak in lmap: ak = lmap[ak]
    ci = ord(ak) - ord('A')
    if ci < 0 or ci >= 4: continue
    rows.append({
        'question_id': f'arc_{i:05d}', 'dataset': 'arc_challenge',
        'subject': 'science', 'question': item['question'],
        'option_a': opts[0], 'option_b': opts[1],
        'option_c': opts[2], 'option_d': opts[3],
        'correct_answer': opts[ci], 'correct_letter': ak,
    })
arc = pd.DataFrame(rows)
print(f'  ARC: {len(arc)} questions')

# ── TruthfulQA ──
print('Loading TruthfulQA...')
ds_tqa = load_dataset('truthfulqa/truthful_qa', 'multiple_choice', split='validation')
rows = []
for i, item in enumerate(ds_tqa):
    mc1 = item['mc1_targets']
    choices = list(mc1['choices'])
    labels = list(mc1['labels'])
    if 1 not in labels: continue
    ci = labels.index(1)
    while len(choices) < 4:
        choices.append('[No option]')
        labels.append(0)
    if ci >= 4:
        choices[3], choices[ci] = choices[ci], choices[3]
        labels[3], labels[ci] = labels[ci], labels[3]
        ci = 3
    rows.append({
        'question_id': f'tqa_{i:05d}', 'dataset': 'truthfulqa',
        'subject': 'misconceptions', 'question': item['question'],
        'option_a': choices[0], 'option_b': choices[1],
        'option_c': choices[2], 'option_d': choices[3],
        'correct_answer': choices[ci], 'correct_letter': LETTER_MAP[ci],
    })
tqa = pd.DataFrame(rows)
print(f'  TQA: {len(tqa)} questions')

full_df = pd.concat([mmlu, arc, tqa], ignore_index=True)
print(f'\nTotal: {len(full_df)} questions')

## 4. Stratified Sample (1000 questions)

In [ ]:
np.random.seed(42)

mmlu_sample = mmlu.groupby('subject', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), max(1, round(800 * len(x) / len(mmlu)))),
                       random_state=42), include_groups=False
)
# Re-add grouping columns that were excluded
mmlu_sample = mmlu.loc[mmlu_sample.index]
if len(mmlu_sample) > 800:
    mmlu_sample = mmlu_sample.sample(n=800, random_state=42)

arc_sample = arc.sample(n=min(100, len(arc)), random_state=42)
tqa_sample = tqa.sample(n=min(100, len(tqa)), random_state=42)

df = pd.concat([mmlu_sample, arc_sample, tqa_sample], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Sample: {len(df)} questions')
print(f'  MMLU: {len(mmlu_sample)} across {mmlu_sample["subject"].nunique()} subjects')
print(f'  ARC:  {len(arc_sample)}')
print(f'  TQA:  {len(tqa_sample)}')

## 5. Annotate Question Features

In [ ]:
# ── Domain mapping ──
STEM = {'abstract_algebra','anatomy','astronomy','college_biology','college_chemistry',
        'college_computer_science','college_mathematics','college_physics','computer_security',
        'conceptual_physics','electrical_engineering','elementary_mathematics',
        'high_school_biology','high_school_chemistry','high_school_computer_science',
        'high_school_mathematics','high_school_physics','high_school_statistics',
        'machine_learning','medical_genetics','virology'}
HUM = {'formal_logic','high_school_european_history','high_school_us_history',
       'high_school_world_history','international_law','jurisprudence',
       'logical_fallacies','moral_disputes','moral_scenarios','philosophy',
       'prehistory','world_religions'}
SOC = {'econometrics','high_school_geography','high_school_government_and_politics',
       'high_school_macroeconomics','high_school_microeconomics','high_school_psychology',
       'human_sexuality','professional_psychology','public_relations','security_studies',
       'sociology','us_foreign_policy'}

def get_domain(row):
    if row['dataset'] == 'arc_challenge': return 'STEM'
    if row['dataset'] == 'truthfulqa': return 'Misconceptions'
    s = row['subject']
    if s in STEM: return 'STEM'
    if s in HUM: return 'Humanities'
    if s in SOC: return 'Social Sciences'
    return 'Professional/Other'

NEG = re.compile(r'\b(NOT|EXCEPT|NEVER|NEITHER|NOR|CANNOT)\b', re.IGNORECASE)
NUM = re.compile(r'\b\d+[\d,]*\.?\d*\s*(?:%|percent|dollars?|km|m|kg|g|years?|days?)?\b')

# Compute features
df['question_length'] = df['question'].str.split().str.len()
df['has_negation'] = df['question'].apply(lambda q: int(bool(NEG.search(q))))
df['has_numerical'] = df.apply(lambda r: int(len(NUM.findall(
    r['question'] + ' ' + ' '.join([str(r[c]) for c in ['option_a','option_b','option_c','option_d']])
)) >= 2), axis=1)
df['domain_category'] = df.apply(get_domain, axis=1)
df['has_misleading_premise'] = (df['dataset'] == 'truthfulqa').astype(int)
df['reasoning_steps'] = pd.qcut(df['question_length'], q=4, labels=[1,2,3,4], duplicates='drop').astype(int)

# Option similarity via sentence-BERT
print('Computing option similarity...')
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=device)
all_opts = []
for _, r in df.iterrows():
    all_opts.extend([str(r['option_a']), str(r['option_b']), str(r['option_c']), str(r['option_d'])])
all_emb = sbert.encode(all_opts, show_progress_bar=True, batch_size=512)
sims = []
for i in range(len(df)):
    e = all_emb[i*4:i*4+4]
    sm_ = cosine_similarity(e)
    sims.append(float(sm_[np.triu_indices(4, k=1)].mean()))
df['option_similarity'] = sims

print(f'\nFeature summary:')
print(f'  question_length: mean={df["question_length"].mean():.1f}')
print(f'  has_negation: {df["has_negation"].sum()} ({100*df["has_negation"].mean():.1f}%)')
print(f'  has_numerical: {df["has_numerical"].sum()} ({100*df["has_numerical"].mean():.1f}%)')
print(f'  has_misleading: {df["has_misleading_premise"].sum()}')
print(f'  option_similarity: mean={df["option_similarity"].mean():.3f}')
print(f'  domains:\n{df["domain_category"].value_counts()}')

## 6. Collect Confidence from All Models

In [ ]:
def parse_response(text):
    """Extract answer letter and confidence from model output."""
    if not text:
        return 'PARSE_ERROR', -1
    text = text.strip()
    # Answer
    m = re.search(r'Answer:\s*\[?([A-Da-d])\]?', text)
    if not m: m = re.search(r'(?:correct answer|answer)\s*(?:is|:)\s*\(?([A-Da-d])\)?', text, re.I)
    if not m: m = re.search(r'^([A-Da-d])\b', text, re.M)
    answer = m.group(1).upper() if m else 'PARSE_ERROR'
    # Confidence
    c = re.search(r'Confidence:\s*\[?(\d{1,3})\]?', text)
    if not c: c = re.search(r'confidence\s*(?:is|:|-|=)\s*\[?(\d{1,3})\]?', text, re.I)
    if not c:
        nums = [int(n) for n in re.findall(r'\b(\d{1,3})\b', text) if 0 <= int(n) <= 100]
        if nums:
            c = type('M', (), {'group': lambda self, x: str(nums[-1])})()
    conf = min(max(int(c.group(1)), 0), 100) if c else -1
    return answer, conf


def build_prompt(row):
    opts = f"A) {row['option_a']}\nB) {row['option_b']}\nC) {row['option_c']}\nD) {row['option_d']}"
    return PROMPT_TEMPLATE.format(question=row['question'], options=opts)


def collect_model(model_name, model_id, df):
    """Load model, run all questions, return results DataFrame."""
    print(f'\n{"="*60}')
    print(f'Loading {model_name} ({model_id})...')
    print(f'{"="*60}')

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.float16, device_map='auto', trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    results = []
    t0 = time.time()

    for i, (_, row) in enumerate(df.iterrows()):
        prompt = build_prompt(row)

        # Format as chat
        messages = [{'role': 'user', 'content': prompt}]
        try:
            text_input = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except:
            text_input = f'<|user|>\n{prompt}<|end|>\n<|assistant|>\n'

        inputs = tokenizer(text_input, return_tensors='pt', truncation=True, max_length=2048).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False, temperature=None, top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated = outputs[0][inputs['input_ids'].shape[1]:]
        raw = tokenizer.decode(generated, skip_special_tokens=True)
        answer, conf = parse_response(raw)

        results.append({
            'question_id': row['question_id'],
            'model': model_name,
            'model_answer': answer,
            'model_confidence': conf,
            'is_correct': int(answer == row['correct_letter']),
            'raw_response': raw,
        })

        if (i+1) % 100 == 0:
            elapsed = time.time() - t0
            rate = (i+1) / elapsed
            eta = (len(df) - i - 1) / rate
            print(f'  [{model_name}] {i+1}/{len(df)} ({rate:.1f} q/s, ETA {eta/60:.0f}m)')

    elapsed = time.time() - t0
    rdf = pd.DataFrame(results)
    valid = rdf[rdf['model_confidence'] >= 0]

    print(f'\n  Done in {elapsed/60:.1f} min')
    print(f'  Accuracy: {valid["is_correct"].mean():.3f}')
    print(f'  Avg confidence: {valid["model_confidence"].mean():.1f}')
    print(f'  Parse errors: {(rdf["model_answer"] == "PARSE_ERROR").sum()}')
    cf = ((valid['model_confidence'] >= 80) & (valid['is_correct'] == 0)).sum()
    print(f'  Confident failures: {cf} ({100*cf/len(valid):.1f}%)')

    # Free GPU memory
    del model, tokenizer
    torch.cuda.empty_cache()

    return rdf

In [ ]:
# Run all models
all_responses = []
for name, mid in MODELS.items():
    rdf = collect_model(name, mid, df)
    rdf.to_parquet(OUT / f'responses_{name}.parquet', index=False)
    all_responses.append(rdf)

responses = pd.concat(all_responses, ignore_index=True)
responses.to_parquet(OUT / 'all_responses.parquet', index=False)
print(f'\nTotal responses: {len(responses)}')

## 7. Analysis

In [ ]:
# Merge features with responses
data = responses.merge(df, on='question_id', how='inner')
data = data[(data['model_answer'] != 'PARSE_ERROR') & (data['model_answer'] != 'ERROR') &
            (data['model_confidence'] >= 0) & (data['model_confidence'] <= 100)]

data['confident_failure'] = ((data['model_confidence'] >= CONFIDENT_THRESHOLD) & (data['is_correct'] == 0)).astype(int)
data['miscalibration'] = abs(data['model_confidence'] / 100.0 - data['is_correct'])

print(f'Valid responses: {len(data)}')
print(f'Confident failures: {data["confident_failure"].sum()} ({100*data["confident_failure"].mean():.1f}%)')

In [ ]:
# ── ECE function ──
def compute_ece(conf, acc, n_bins=10):
    boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (conf >= boundaries[i]) & (conf < boundaries[i+1])
        if i == n_bins - 1: mask = (conf >= boundaries[i]) & (conf <= boundaries[i+1])
        if mask.sum() == 0: continue
        ece += mask.sum() / len(conf) * abs(acc[mask].mean() - conf[mask].mean())
    return ece

# ── Analysis 1: Descriptive stats per model ──
print('='*60)
print('ANALYSIS 1: Descriptive Statistics')
print('='*60)
desc_rows = []
for model in data['model'].unique():
    m = data[data['model'] == model]
    c = m['model_confidence'].values / 100.0
    a = m['is_correct'].values
    ece = compute_ece(c, a)
    row = {
        'model': model, 'n': len(m), 'accuracy': a.mean(),
        'avg_confidence': c.mean()*100, 'ECE': ece,
        'cf_rate': m['confident_failure'].mean(),
        'cf_count': m['confident_failure'].sum(),
    }
    desc_rows.append(row)
    print(f"\n  {model}: acc={row['accuracy']:.3f}, avg_conf={row['avg_confidence']:.1f}, ECE={ece:.4f}, CF={row['cf_count']} ({100*row['cf_rate']:.1f}%)")

desc_df = pd.DataFrame(desc_rows)
desc_df.to_csv(OUT / 'descriptive_stats.csv', index=False)

In [ ]:
# ── Analysis 2: CF rates by feature ──
print('='*60)
print('ANALYSIS 2: Confident Failure Rates by Feature')
print('='*60)

for col in ['has_negation', 'has_numerical', 'has_misleading_premise']:
    t = data.groupby(col).agg(n=('confident_failure','count'),
                               cf_rate=('confident_failure','mean'),
                               accuracy=('is_correct','mean')).round(4)
    print(f'\n  {col}:\n{t}')

t = data.groupby('domain_category').agg(
    n=('confident_failure','count'), cf_rate=('confident_failure','mean'),
    accuracy=('is_correct','mean')).round(4).sort_values('cf_rate', ascending=False)
print(f'\n  domain_category:\n{t}')

t = data.groupby('reasoning_steps').agg(
    n=('confident_failure','count'), cf_rate=('confident_failure','mean'),
    accuracy=('is_correct','mean')).round(4)
print(f'\n  reasoning_steps:\n{t}')

In [ ]:
# ── Analysis 3: Logistic Regression ──
print('='*60)
print('ANALYSIS 3: Logistic Regression — Predicting Confident Failure')
print('='*60)

PRED_COLS = ['question_length','has_negation','has_numerical','option_similarity',
             'reasoning_steps','has_misleading_premise']

mdf = data.copy()
mdf = pd.get_dummies(mdf, columns=['domain_category'], drop_first=True, dtype=int)
mdf = pd.get_dummies(mdf, columns=['model'], drop_first=True, dtype=int)
dom_d = [c for c in mdf.columns if c.startswith('domain_category_')]
mod_d = [c for c in mdf.columns if c.startswith('model_')]
predictors = PRED_COLS + dom_d + mod_d
predictors = [p for p in predictors if p in mdf.columns]

adf = mdf[predictors + ['confident_failure']].dropna()
X = adf[predictors]
y = adf['confident_failure']
print(f'  N={len(adf)}, events={y.sum()} ({100*y.mean():.1f}%)')

X_c = sm.add_constant(X)
logit = sm.Logit(y, X_c).fit(disp=0, maxiter=100)
print(logit.summary2().tables[1])
print(f'\n  Pseudo R² (McFadden): {logit.prsquared:.4f}')

coef = pd.DataFrame({
    'variable': logit.params.index, 'coefficient': logit.params.values,
    'p_value': logit.pvalues.values, 'odds_ratio': np.exp(logit.params.values),
})
coef.to_csv(OUT / 'logistic_regression_coefficients.csv', index=False)
print('\nSignificant predictors (p<0.05):')
sig = coef[(coef['p_value'] < 0.05) & (coef['variable'] != 'const')]
for _, r in sig.iterrows():
    d = '+' if r['coefficient'] > 0 else '-'
    print(f"  {d} {r['variable']}: OR={r['odds_ratio']:.3f}, p={r['p_value']:.4f}")

In [ ]:
# ── Analysis 4: Per-model regressions (RQ2) ──
print('='*60)
print('ANALYSIS 4: Per-Model Failure Profiles')
print('='*60)

for model in data['model'].unique():
    m = data[data['model'] == model].copy()
    m = pd.get_dummies(m, columns=['domain_category'], drop_first=True, dtype=int)
    dd = [c for c in m.columns if c.startswith('domain_category_')]
    preds = PRED_COLS + dd
    preds = [p for p in preds if p in m.columns]
    a = m[preds + ['confident_failure']].dropna()
    if a['confident_failure'].sum() < 10:
        print(f'  {model}: too few CF ({a["confident_failure"].sum()}), skipping')
        continue
    Xm = sm.add_constant(a[preds])
    ym = a['confident_failure']
    try:
        lg = sm.Logit(ym, Xm).fit(disp=0, maxiter=100)
        print(f'\n  {model} (N={len(a)}, CF={ym.sum()}, R²={lg.prsquared:.4f}):')
        for var in lg.pvalues[lg.pvalues < 0.05].index:
            if var == 'const': continue
            print(f"    {'+'if lg.params[var]>0 else '-'} {var}: OR={np.exp(lg.params[var]):.3f}, p={lg.pvalues[var]:.4f}")
    except:
        print(f'  {model}: regression failed')

In [ ]:
# ── Analysis 5: Random Forest ──
print('='*60)
print('ANALYSIS 5: Random Forest Feature Importance')
print('='*60)

le_dom = LabelEncoder()
le_mod = LabelEncoder()
rf_df = data.copy()
rf_df['domain_encoded'] = le_dom.fit_transform(rf_df['domain_category'])
rf_df['model_encoded'] = le_mod.fit_transform(rf_df['model'])
feat_cols = PRED_COLS + ['domain_encoded', 'model_encoded']
feat_cols = [c for c in feat_cols if c in rf_df.columns]
a = rf_df[feat_cols + ['confident_failure']].dropna()
Xrf, yrf = a[feat_cols], a['confident_failure']

rf = RandomForestClassifier(n_estimators=500, max_depth=10, min_samples_leaf=20,
                            class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(Xrf, yrf)

imp = pd.DataFrame({'feature': feat_cols, 'importance': rf.feature_importances_}
                   ).sort_values('importance', ascending=False)
print('\nFeature Importance (Gini):')
for _, r in imp.iterrows():
    bar = '█' * int(r['importance'] * 100)
    print(f"  {r['feature']:30s} {r['importance']:.4f}  {bar}")
imp.to_csv(OUT / 'random_forest_importance.csv', index=False)

cv = cross_val_score(rf, Xrf, yrf, cv=5, scoring='roc_auc')
print(f'\n  5-Fold CV AUC: {cv.mean():.4f} (+/- {cv.std():.4f})')

In [ ]:
# ── Analysis 6: Per-domain ECE ──
print('='*60)
print('ANALYSIS 6: Per-Domain ECE')
print('='*60)

dom_rows = []
for dom in data['domain_category'].unique():
    d = data[data['domain_category'] == dom]
    c = d['model_confidence'].values / 100.0
    a = d['is_correct'].values
    dom_rows.append({'domain': dom, 'n': len(d), 'accuracy': a.mean(),
                     'avg_conf': c.mean(), 'ECE': compute_ece(c, a),
                     'cf_rate': d['confident_failure'].mean()})
dom_df = pd.DataFrame(dom_rows).sort_values('ECE', ascending=False)
print(dom_df.to_string(index=False))
dom_df.to_csv(OUT / 'domain_ece.csv', index=False)

## 8. Figures

In [ ]:
sns.set_theme(style='whitegrid', font_scale=1.1)
models = data['model'].unique()
n_models = len(models)

# ── Fig 1: Calibration curves ──
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))
if n_models == 1: axes = [axes]
for ax, model in zip(axes, models):
    m = data[data['model'] == model]
    c = m['model_confidence'].values / 100.0
    a = m['is_correct'].values
    bins = np.linspace(0, 1, 11)
    centers, accs = [], []
    for i in range(10):
        mask = (c >= bins[i]) & (c < bins[i+1]) if i < 9 else (c >= bins[i]) & (c <= bins[i+1])
        if mask.sum() > 0:
            centers.append(c[mask].mean())
            accs.append(a[mask].mean())
    ax.plot([0,1],[0,1],'k--',alpha=0.5)
    ax.bar(centers, accs, width=0.08, alpha=0.7, color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_title(f'{model}\nECE={compute_ece(c,a):.4f}')
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig(FIG / 'fig1_calibration_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig1')

# ── Fig 2: CF rate by feature ──
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
# Reasoning steps
r = data.groupby('reasoning_steps')['confident_failure'].mean()
axes[0,0].bar(r.index.astype(str), r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[0,0].set_xlabel('Reasoning Steps'); axes[0,0].set_ylabel('CF Rate'); axes[0,0].set_title('By Reasoning Steps')
# Negation
r = data.groupby('has_negation')['confident_failure'].mean()
axes[0,1].bar(['No','Yes'], r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[0,1].set_ylabel('CF Rate'); axes[0,1].set_title('By Negation')
# Misleading
r = data.groupby('has_misleading_premise')['confident_failure'].mean()
axes[0,2].bar(['No','Yes'], r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[0,2].set_ylabel('CF Rate'); axes[0,2].set_title('By Misleading Premise')
# Domain
r = data.groupby('domain_category')['confident_failure'].mean().sort_values()
axes[1,0].barh(r.index, r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[1,0].set_xlabel('CF Rate'); axes[1,0].set_title('By Domain')
# Option similarity
data['os_q'] = pd.qcut(data['option_similarity'], 4, labels=['Q1\n(low)','Q2','Q3','Q4\n(high)'], duplicates='drop')
r = data.groupby('os_q')['confident_failure'].mean()
axes[1,1].bar(r.index.astype(str), r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[1,1].set_xlabel('Option Similarity'); axes[1,1].set_ylabel('CF Rate'); axes[1,1].set_title('By Option Similarity')
# Numerical
r = data.groupby('has_numerical')['confident_failure'].mean()
axes[1,2].bar(['No','Yes'], r.values, color='coral', edgecolor='black', linewidth=0.5)
axes[1,2].set_ylabel('CF Rate'); axes[1,2].set_title('By Numerical Content')
plt.suptitle('Confident Failure Rate by Question Feature', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'fig2_cf_rate_by_feature.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig2')

# ── Fig 3: Confidence distributions ──
fig, axes = plt.subplots(1, n_models, figsize=(5*n_models, 5))
if n_models == 1: axes = [axes]
for ax, model in zip(axes, models):
    m = data[data['model'] == model]
    ax.hist(m[m['is_correct']==1]['model_confidence'], bins=20, alpha=0.6, label='Correct', color='green', density=True)
    ax.hist(m[m['is_correct']==0]['model_confidence'], bins=20, alpha=0.6, label='Incorrect', color='red', density=True)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Density'); ax.set_title(model); ax.legend()
plt.suptitle('Confidence Distributions: Correct vs Incorrect', fontsize=14)
plt.tight_layout()
plt.savefig(FIG / 'fig3_confidence_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig3')

# ── Fig 4: Feature importance ──
imp_s = imp.sort_values('importance', ascending=True)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_s['feature'], imp_s['importance'], color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Feature Importance (Gini)'); ax.set_title('Random Forest: Predictors of Confident Failure')
plt.tight_layout()
plt.savefig(FIG / 'fig4_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig4')

# ── Fig 5: Heatmap ──
pivot = data.pivot_table(values='confident_failure', index='domain_category', columns='model', aggfunc='mean')
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Confident Failure Rate: Model x Domain')
plt.tight_layout()
plt.savefig(FIG / 'fig5_model_domain_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved fig5')

## 9. Download Results

In [ ]:
# Save everything to a zip for download
import shutil

# Save the merged analysis data
data.to_csv(OUT / 'full_analysis_data.csv', index=False)
df.to_csv(OUT / 'questions_annotated.csv', index=False)

shutil.make_archive('experiment_results', 'zip', '.', 'results')
shutil.make_archive('experiment_figures', 'zip', '.', 'figures')

print('Download these files:')
print('  - experiment_results.zip (all CSVs and parquet files)')
print('  - experiment_figures.zip (all PNG figures)')

# In Colab, use:
try:
    from google.colab import files
    files.download('experiment_results.zip')
    files.download('experiment_figures.zip')
except:
    print('Not in Colab - files saved locally')